# 🧪 Colab 1 — Build a Financial RAG System
### Day 15 · Gen AI APIs · Enterprise AI Engineering Series

---

## 🎯 Objective
Build an end-to-end Retrieval-Augmented Generation (RAG) pipeline that:
- Ingests and chunks SEC 10-K filings
- Embeds documents using HuggingFace `all-MiniLM-L6-v2`
- Stores vectors in FAISS
- Generates answers using **Azure OpenAI GPT-4o**
- Evaluates quality with **RAGAS** metrics

## 📦 Industry Scenario: FinSight AI
> You are an engineer at a Tier-1 investment bank building *FinSight*, an AI Research Analyst Assistant.
> Analysts need to query annual reports in natural language. Your goal: faithfulness ≥ 0.85, latency < 3s, cost < $0.002/query.

## ⏱️ Estimated Time: 60–75 minutes

---

## ⚙️ Step 0 — Install Dependencies

In [1]:
!pip install -q langchain langchain-community langchain-openai \
    faiss-cpu sentence-transformers openai \
    ragas datasets pypdf tiktoken \
    python-dotenv tqdm rich
print('✅ Dependencies installed')

✅ Dependencies installed


## 🔑 Step 1 — Configure Credentials

> **In Colab:** use `Secrets` (🔑 icon in left sidebar) to store keys safely — never hardcode them.

In [2]:
import os
from google.colab import userdata

# --- Azure OpenAI Configuration ---
# Set these in Colab Secrets: AZURE_OPENAI_ENDPOINT, AZURE_OPENAI_KEY, AZURE_OPENAI_DEPLOYMENT
AZURE_ENDPOINT   = userdata.get('AZURE_OPENAI_ENDPOINT')   # e.g. https://myresource.openai.azure.com/
AZURE_API_KEY    = userdata.get('AZURE_OPENAI_KEY')
AZURE_DEPLOYMENT = userdata.get('AZURE_OPENAI_DEPLOYMENT')  # e.g. 'gpt-4o'
AZURE_API_VERSION = '2024-06-01'

# --- HuggingFace Token (for private models / higher rate limits) ---
# HF_TOKEN = userdata.get('HF_TOKEN')  # optional for public models

print('✅ Credentials loaded' if AZURE_API_KEY else '❌ Missing Azure credentials — check Secrets')

✅ Credentials loaded


## 📄 Step 2 — Load & Inspect SEC 10-K Filings

We'll use 3 Apple 10-K filing excerpts. In production, you'd ingest thousands of PDFs from an Azure Blob Storage container.

In [3]:
# ─── For demo purposes: inline sample financial documents ─────────────────────
# In production: load from Azure Blob Storage or a PDF directory

SAMPLE_DOCS = [
    {
        "source": "Apple_10K_2023_Risk",
        "text": """RISK FACTORS
Apple's operations and financial results are subject to various risks and uncertainties.
Global and regional economic conditions, including conditions resulting from financial and credit market fluctuations,
can adversely affect demand for Apple's products and services.
Apple faces intense competition in all of its business areas from well-established companies with significant
resources, as well as from new market entrants.
Apple depends on the performance of distributors, carriers, wholesalers and other resellers.
The Company's fiscal year 2023 revenue was $383.3 billion, compared to $394.3 billion in fiscal 2022,
a decrease of approximately 2.8 percent.
The Company's net income for fiscal 2023 was $97.0 billion, or $6.13 diluted earnings per share,
compared to $99.8 billion, or $6.11 diluted earnings per share, in fiscal 2022.
Apple's gross margin percentage was 44.1% in fiscal 2023, compared to 43.3% in fiscal 2022.
Services revenue reached an all-time high of $85.2 billion in fiscal 2023, up 9 percent year over year."""
    },
    {
        "source": "Apple_10K_2023_Products",
        "text": """PRODUCTS AND SERVICES
Apple designs, manufactures and markets smartphones, personal computers, tablets, wearables and accessories.
iPhone is Apple's line of smartphones based on its iOS operating system.
iPhone net sales were $200.6 billion in fiscal 2023, representing approximately 52% of total revenue.
Mac net sales were $29.4 billion in fiscal 2023, down from $40.2 billion in fiscal 2022.
iPad net sales were $28.3 billion in fiscal 2023.
Wearables, Home and Accessories net sales were $39.8 billion in fiscal 2023.
Apple's Services segment includes advertising, AppleCare, cloud, digital content, payment and other services.
The App Store, Apple Music, Apple TV+, Apple Arcade, iCloud and Apple Pay are key Services offerings.
The Company had approximately 2.2 billion active devices at the end of fiscal year 2023."""
    },
    {
        "source": "Apple_10K_2023_Liquidity",
        "text": """LIQUIDITY AND CAPITAL RESOURCES
The Company believes its existing balances of cash, cash equivalents and unrestricted marketable securities,
together with cash generated by operations, will be sufficient to satisfy its expected cash needs.
Cash and cash equivalents as of September 30, 2023 were $29.965 billion.
Total marketable securities were $100.544 billion, consisting of current marketable securities of $31.590 billion
and non-current marketable securities of $100.544 billion.
During fiscal 2023, the Company returned over $77 billion to shareholders,
including $15.1 billion in dividends and dividend equivalents and $62.2 billion through repurchases of 471 million shares.
Capital expenditures were $10.959 billion in fiscal 2023.
The Company's long-term debt as of September 30, 2023 was $95.281 billion."""
    },
]

print(f'📂 Loaded {len(SAMPLE_DOCS)} documents')
for doc in SAMPLE_DOCS:
    print(f'  - {doc["source"]}: {len(doc["text"])} characters')

📂 Loaded 3 documents
  - Apple_10K_2023_Risk: 1050 characters
  - Apple_10K_2023_Products: 822 characters
  - Apple_10K_2023_Liquidity: 816 characters


## ✂️ Step 3 — Chunking Strategy

Chunking is one of the most impactful RAG design decisions. We'll experiment with different chunk sizes.

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

def create_chunks(docs: list, chunk_size: int = 512, chunk_overlap: int = 64) -> list:
    """Convert raw documents to LangChain Document chunks."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
        length_function=len,
    )
    lc_docs = [Document(page_content=d["text"], metadata={"source": d["source"]}) for d in docs]
    chunks = splitter.split_documents(lc_docs)
    return chunks

# Default configuration
CHUNK_SIZE = 512
CHUNK_OVERLAP = 64
chunks = create_chunks(SAMPLE_DOCS, CHUNK_SIZE, CHUNK_OVERLAP)

print(f'✅ Created {len(chunks)} chunks (size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP})')
print('\n📋 Sample chunk:')
print(f'  Source: {chunks[0].metadata["source"]}')
print(f'  Content: {chunks[0].page_content[:200]}...')

✅ Created 7 chunks (size=512, overlap=64)

📋 Sample chunk:
  Source: Apple_10K_2023_Risk
  Content: RISK FACTORS
Apple's operations and financial results are subject to various risks and uncertainties.
Global and regional economic conditions, including conditions resulting from financial and credit ...


## 🧮 Step 4 — Embed with HuggingFace `all-MiniLM-L6-v2`

This 80M-parameter model produces 384-dimensional embeddings. It's fast, free, and surprisingly capable for financial text.

In [5]:
import time
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Load the embedding model (downloads ~90MB on first run)
print('⏳ Loading HuggingFace embedding model...')
embedding_model = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs={'device': 'cpu'},  # change to 'cuda' if GPU available
    encode_kwargs={'normalize_embeddings': True, 'batch_size': 32}
)

# Build FAISS index
print('⏳ Building FAISS index...')
t0 = time.time()
vectorstore = FAISS.from_documents(chunks, embedding_model)
elapsed = time.time() - t0

print(f'✅ FAISS index built in {elapsed:.2f}s')
print(f'   Vectors: {vectorstore.index.ntotal}')
print(f'   Dimension: {vectorstore.index.d}')

⏳ Loading HuggingFace embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

⏳ Building FAISS index...
✅ FAISS index built in 0.69s
   Vectors: 7
   Dimension: 384


## 🔍 Step 5 — Build the Retriever

We'll use Maximal Marginal Relevance (MMR) to balance relevance with diversity in retrieved chunks.

In [6]:
# MMR retriever: fetch_k candidates, return top k diverse results
retriever = vectorstore.as_retriever(
    search_type='mmr',
    search_kwargs={
        'k': 4,          # number of chunks to return
        'fetch_k': 10,   # candidate pool for MMR
        'lambda_mult': 0.7  # 1.0 = pure relevance, 0.0 = pure diversity
    }
)

# Test retrieval
test_query = 'What was Apple revenue in 2023?'
results = retriever.invoke(test_query)
print(f'🔍 Query: "{test_query}"')
print(f'   Retrieved {len(results)} chunks:')
for i, r in enumerate(results):
    print(f'   [{i+1}] {r.metadata["source"]}: {r.page_content[:100]}...')

🔍 Query: "What was Apple revenue in 2023?"
   Retrieved 4 chunks:
   [1] Apple_10K_2023_Risk: resources, as well as from new market entrants.
Apple depends on the performance of distributors, ca...
   [2] Apple_10K_2023_Risk: Apple's gross margin percentage was 44.1% in fiscal 2023, compared to 43.3% in fiscal 2022.
Services...
   [3] Apple_10K_2023_Products: iPad net sales were $28.3 billion in fiscal 2023.
Wearables, Home and Accessories net sales were $39...
   [4] Apple_10K_2023_Products: PRODUCTS AND SERVICES
Apple designs, manufactures and markets smartphones, personal computers, table...


## 🤖 Step 6 — Azure OpenAI Generation

Connect GPT-4o via Azure OpenAI to synthesise answers from retrieved context.

In [7]:
from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Initialise Azure OpenAI LLM
llm = AzureChatOpenAI(
    azure_endpoint=AZURE_ENDPOINT,
    azure_deployment=AZURE_DEPLOYMENT,
    openai_api_version=AZURE_API_VERSION,
    openai_api_key=AZURE_API_KEY,
    temperature=0,          # deterministic for financial queries
    max_tokens=512,
    timeout=30,
)

# RAG prompt template — cite sources, be conservative with numbers
RAG_PROMPT = ChatPromptTemplate.from_template("""
You are FinSight, an AI research analyst for a Tier-1 investment bank.
Answer the analyst's question ONLY using the provided context.
If the context does not contain the answer, say: "Insufficient information in the retrieved context."
Always cite the specific source document at the end of your answer.

CONTEXT:
{context}

ANALYST QUESTION: {question}

ANSWER (cite source):
""")

def format_docs(docs):
    return "\n\n".join(
        f"[Source: {d.metadata['source']}]\n{d.page_content}"
        for d in docs
    )

# Build RAG chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

print('✅ RAG chain built')

✅ RAG chain built


## 🧪 Step 7 — Run Queries & Measure Latency

In [8]:
import time

TEST_QUERIES = [
    'What was Apple\'s total revenue in fiscal year 2023?',
    'How much cash did Apple have at the end of fiscal 2023?',
    'What percentage of Apple\'s revenue came from iPhone in 2023?',
    'How much did Apple return to shareholders in fiscal 2023?',
    'What is Apple\'s gross margin for fiscal 2023?',
]

results_log = []

for query in TEST_QUERIES:
    t0 = time.time()
    answer = rag_chain.invoke(query)
    latency = time.time() - t0
    results_log.append({'query': query, 'answer': answer, 'latency_s': round(latency, 2)})
    print(f'❓ {query}')
    print(f'   ⏱️  {latency:.2f}s')
    print(f'   💬 {answer[:200]}...')
    print()

❓ What was Apple's total revenue in fiscal year 2023?
   ⏱️  1.75s
   💬 Apple's total revenue in fiscal year 2023 was $383.3 billion. [Source: Apple_10K_2023_Risk]...

❓ How much cash did Apple have at the end of fiscal 2023?
   ⏱️  1.08s
   💬 Insufficient information in the retrieved context....

❓ What percentage of Apple's revenue came from iPhone in 2023?
   ⏱️  3.13s
   💬 iPhone net sales were $200.6 billion in fiscal 2023, and Apple's total revenue was $383.3 billion. Therefore, the percentage of Apple's revenue that came from iPhone in 2023 is approximately **52%**.
...

❓ How much did Apple return to shareholders in fiscal 2023?
   ⏱️  1.54s
   💬 Apple returned over $77 billion to shareholders in fiscal 2023, including $15.1 billion in dividends and dividend equivalents and $62.2 billion through repurchases of 471 million shares. [Source: Appl...

❓ What is Apple's gross margin for fiscal 2023?
   ⏱️  1.37s
   💬 Apple's gross margin percentage for fiscal 2023 was 44.1%. [Source

## 📊 Step 8 — RAGAS Evaluation

[RAGAS](https://docs.ragas.io) evaluates RAG pipelines on 4 key metrics:

| Metric | What it measures |
|--------|------------------|
| **Faithfulness** | Are answer claims grounded in retrieved context? |
| **Answer Relevance** | Does the answer address the question? |
| **Context Recall** | Does retrieved context cover what's needed? |
| **Context Precision** | Is the context precise (minimal noise)? |

In [9]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_recall, context_precision
from langchain_openai import AzureOpenAIEmbeddings

# Ground truth answers for evaluation
GROUND_TRUTHS = [
    'Apple\'s total revenue in fiscal 2023 was $383.3 billion.',
    'Apple had $29.965 billion in cash and cash equivalents at end of fiscal 2023.',
    'iPhone represented approximately 52% of Apple\'s total revenue in fiscal 2023.',
    'Apple returned over $77 billion to shareholders in fiscal 2023.',
    'Apple\'s gross margin was 44.1% in fiscal 2023.',
]

# Collect contexts used per query
contexts_used = []
for q in TEST_QUERIES:
    docs = retriever.invoke(q)
    contexts_used.append([d.page_content for d in docs])

# Build RAGAS evaluation dataset
eval_dataset = Dataset.from_dict({
    'question':  TEST_QUERIES,
    'answer':    [r['answer'] for r in results_log],
    'contexts':  contexts_used,
    'ground_truth': GROUND_TRUTHS,
})

# Azure OpenAI embeddings for RAGAS
az_embeddings = AzureOpenAIEmbeddings(
    azure_endpoint=AZURE_ENDPOINT,
    azure_deployment='text-embedding-ada-002',  # must be deployed in your Azure resource
    openai_api_key=AZURE_API_KEY,
    openai_api_version=AZURE_API_VERSION,
)

# Run RAGAS evaluation
print('⏳ Running RAGAS evaluation...')
ragas_results = evaluate(
    dataset=eval_dataset,
    metrics=[faithfulness, answer_relevancy, context_recall, context_precision],
    llm=llm,
    embeddings=az_embeddings,
)

print('\n📊 RAGAS Evaluation Results:')
print(ragas_results)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/tmp/ipykernel_15825/2413967300.py:3: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_recall, context_precision
/tmp/ipykernel_15825/2413967300.py:3: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metri

⏳ Running RAGAS evaluation...


Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[5]: NotFoundError(Error code: 404 - {'error': {'code': 'DeploymentNotFound', 'message': 'The API deployment for this resource does not exist. If you created the deployment within the last 5 minutes, please wait a moment and try again.'}})
ERROR:ragas.executor:Exception raised in Job[13]: NotFoundError(Error code: 404 - {'error': {'code': 'DeploymentNotFound', 'message': 'The API deployment for this resource does not exist. If you created the deployment within the last 5 minutes, please wait a moment and try again.'}})
ERROR:ragas.executor:Exception raised in Job[9]: NotFoundError(Error code: 404 - {'error': {'code': 'DeploymentNotFound', 'message': 'The API deployment for this resource does not exist. If you created the deployment within the last 5 minutes, please wait a moment and try again.'}})
ERROR:ragas.executor:Exception raised in Job[1]: NotFoundError(Error code: 404 - {'error': {'code': 'DeploymentNotFound', 'message': 'The API deplo


📊 RAGAS Evaluation Results:
{'faithfulness': 0.8000, 'answer_relevancy': nan, 'context_recall': 0.8000, 'context_precision': 0.6000}


## 🔬 Step 9 — Chunk Size Experiment

Compare RAG quality across different chunk sizes. This is a key hyperparameter.

In [12]:
import pandas as pd

# ⚠️ This cell makes multiple API calls — monitor your Azure token budget
CHUNK_SIZES_TO_TEST = [256, 512, 1024]
experiment_results = []

for cs in CHUNK_SIZES_TO_TEST:
    print(f'\n🔧 Testing chunk_size={cs}...')

    # Rebuild index with new chunk size
    exp_chunks = create_chunks(SAMPLE_DOCS, chunk_size=cs, chunk_overlap=cs//8)
    exp_vs = FAISS.from_documents(exp_chunks, embedding_model)
    exp_retriever = exp_vs.as_retriever(search_type='mmr', search_kwargs={'k': 4, 'fetch_k': 10})

    # Build and run chain
    exp_chain = (
        {"context": exp_retriever | format_docs, "question": RunnablePassthrough()}
        | RAG_PROMPT | llm | StrOutputParser()
    )

    answers, latencies = [], []
    for q in TEST_QUERIES[:3]:  # run subset to save cost
        t0 = time.time()
        a = exp_chain.invoke(q)
        latencies.append(time.time() - t0)
        answers.append(a)

    experiment_results.append({
        'chunk_size': cs,
        'n_chunks': len(exp_chunks),
        'avg_latency_s': round(sum(latencies)/len(latencies), 2),
    })
    print(f'   Chunks: {len(exp_chunks)}, Avg latency: {experiment_results[-1]["avg_latency_s"]}s')

df = pd.DataFrame(experiment_results)
print('\n📊 Chunk Size Comparison:')
print(df.to_string(index=False))
print('\n💡 Tip: Run full RAGAS evaluation per chunk_size to see faithfulness trade-offs')


🔧 Testing chunk_size=256...
   Chunks: 13, Avg latency: 1.68s

🔧 Testing chunk_size=512...
   Chunks: 7, Avg latency: 2.4s

🔧 Testing chunk_size=1024...
   Chunks: 4, Avg latency: 1.88s

📊 Chunk Size Comparison:
 chunk_size  n_chunks  avg_latency_s
        256        13           1.68
        512         7           2.40
       1024         4           1.88

💡 Tip: Run full RAGAS evaluation per chunk_size to see faithfulness trade-offs


## ✅ Summary & Reflection

Complete the following reflection questions:

1. What faithfulness score did you achieve? How does it compare to the FinSight target of ≥ 0.85?
2. Which chunk size gave the best trade-off between faithfulness and latency?
3. What types of queries failed? Why might the RAG pipeline struggle with them?
4. How would you adapt this pipeline for 10,000 PDFs ingested daily?

---

## 🚀 Extension Task — Hybrid Retrieval + Re-Ranker

**Goal:** Improve retrieval precision by combining dense vectors (all-MiniLM) with BM25 sparse retrieval, then re-rank with a cross-encoder.

**Steps:**
1. Install: `!pip install rank_bm25 langchain_community`
2. Build a `BM25Retriever` from the same chunks
3. Use `EnsembleRetriever(retrievers=[dense, bm25], weights=[0.6, 0.4])`
4. Add cross-encoder re-ranker: `cross-encoder/ms-marco-MiniLM-L-6-v2`
5. Re-run RAGAS — target faithfulness > 0.88
6. **Present:** Plot faithfulness vs latency for dense vs hybrid vs hybrid+rerank


In [ ]:
# ─── Extension: Hybrid Retrieval ─────────────────────────────────────────────
# Uncomment and complete this block

# !pip install -q rank_bm25

# from langchain_community.retrievers import BM25Retriever
# from langchain.retrievers import EnsembleRetriever

# bm25_retriever = BM25Retriever.from_documents(chunks)
# bm25_retriever.k = 4

# hybrid_retriever = EnsembleRetriever(
#     retrievers=[retriever, bm25_retriever],
#     weights=[0.6, 0.4]  # 60% dense, 40% BM25
# )

# # Test hybrid retrieval
# hybrid_results = hybrid_retriever.invoke('Apple net income fiscal 2023')
# print(f'Hybrid retrieved {len(hybrid_results)} chunks')
# for r in hybrid_results:
#     print(f'  - {r.metadata["source"]}: {r.page_content[:80]}...')

print('Extension task: complete the hybrid retrieval implementation above!')

In [14]:
!pip install -q rank_bm25

In [15]:
# ─── Extension: Hybrid Retrieval ─────────────────────────────────────────────

# !pip install -q rank_bm25

from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever

# BM25 retriever
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 4

# Hybrid retriever: Dense (FAISS) + Sparse (BM25)
hybrid_retriever = EnsembleRetriever(
    retrievers=[retriever, bm25_retriever],
    weights=[0.6, 0.4]  # 60% dense, 40% BM25
)

# Test hybrid retrieval
query = "Apple net income fiscal 2023"
hybrid_results = hybrid_retriever.invoke(query)

print(f"🔍 Query: {query}")
print(f"Hybrid retrieved {len(hybrid_results)} chunks\n")

for i, r in enumerate(hybrid_results, 1):
    print(f"[{i}] {r.metadata['source']}")
    print(f"    {r.page_content[:100]}...\n")

🔍 Query: Apple net income fiscal 2023
Hybrid retrieved 5 chunks

[1] Apple_10K_2023_Risk
    resources, as well as from new market entrants.
Apple depends on the performance of distributors, ca...

[2] Apple_10K_2023_Products
    PRODUCTS AND SERVICES
Apple designs, manufactures and markets smartphones, personal computers, table...

[3] Apple_10K_2023_Products
    iPad net sales were $28.3 billion in fiscal 2023.
Wearables, Home and Accessories net sales were $39...

[4] Apple_10K_2023_Risk
    Apple's gross margin percentage was 44.1% in fiscal 2023, compared to 43.3% in fiscal 2022.
Services...

[5] Apple_10K_2023_Liquidity
    and non-current marketable securities of $100.544 billion.
During fiscal 2023, the Company returned ...



Extension Task – Hybrid Retrieval

 The hybrid retriever successfully retrieved five relevant chunks from multiple sections of Apple's 2023 10-K report, including Product Information, Risk Factors, and Liquidity data.

The results demonstrate that hybrid retrieval can improve retrieval coverage by combining semantic matching from vector embeddings with keyword-based matching from BM25. This approach has the potential to improve context recall and answer quality compared to using dense retrieval alone.